# Notebook 1: Data preparation

**Paper:** From Keywords to LLM Re-Ranking: A Reproducible, Cost-Aware Benchmark for Resume–Job Matching on Public Data

This notebook:
1. loads the public **resume-job-description-fit** dataset (Hugging Face)
2. explores it (sizes, labels, text lengths)
3. cleans the text and gives every job and resume a stable id
4. checks for duplicates and conflicting labels
5. makes a **train / validation / test split grouped by job**, so no job leaks between splits
6. builds the per-job ranking lists used for NDCG, P@k and MRR
7. exports a **blind sample of 150 pairs** for human labelling (RQ3)

Everything is saved to Google Drive, in `MyDrive/resume-job-matching/data`, so the later notebooks can reuse it.

**Run with:** Runtime → Run all. The first cell asks for permission to use Google Drive.

In [ ]:
# Setup: mount Google Drive and install the one extra library
from google.colab import drive
drive.mount('/content/drive')

!pip -q install datasets

import os, re, json, hashlib
import pandas as pd
import numpy as np

OUT = '/content/drive/MyDrive/resume-job-matching/data'
os.makedirs(OUT, exist_ok=True)
SEED = 42
print('Saving outputs to', OUT)

## 1. Load the dataset

In [ ]:
from datasets import load_dataset

ds = load_dataset('cnamuangtoun/resume-job-description-fit')
print(ds)
raw_train = ds['train'].to_pandas()
raw_test = ds['test'].to_pandas()
raw_train.head(3)

## 2. Explore

In [ ]:
for name, df in [('train', raw_train), ('test', raw_test)]:
    print(f"{name}: {len(df)} pairs | {df.job_description_text.nunique()} unique jobs | "
          f"{df.resume_text.nunique()} unique resumes")
    print(df.label.value_counts().to_string(), '\n')

## 3. Clean the text and assign ids
Cleaning is kept light on purpose: whitespace is normalised, and nothing else changes. Every method is compared on the same text, so heavy preprocessing would favour some methods over others.

Relevance grades for ranking: **Good Fit = 2, Potential Fit = 1, No Fit = 0**.

In [ ]:
REL = {'No Fit': 0, 'Potential Fit': 1, 'Good Fit': 2}

def clean(text):
    text = str(text).replace('\u00a0', ' ')
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def make_id(text, prefix):
    return prefix + hashlib.md5(text.encode('utf-8')).hexdigest()[:10]

def prepare(df, split):
    out = pd.DataFrame({
        'job_text': df.job_description_text.map(clean),
        'resume_text': df.resume_text.map(clean),
        'label': df.label,
    })
    out['job_id'] = out.job_text.map(lambda t: make_id(t, 'J'))
    out['resume_id'] = out.resume_text.map(lambda t: make_id(t, 'R'))
    out['relevance'] = out.label.map(REL)
    out['source_split'] = split
    return out[['job_id', 'resume_id', 'job_text', 'resume_text', 'label', 'relevance', 'source_split']]

pairs = pd.concat([prepare(raw_train, 'train'), prepare(raw_test, 'test')], ignore_index=True)
assert pairs.relevance.notna().all(), 'unexpected label value'
print(pairs.shape)
pairs.head(3)

## 4. Duplicates and conflicting labels

In [ ]:
keys = ['job_id', 'resume_id']
dup_mask = pairs.duplicated(keys, keep=False)
label_counts = pairs[dup_mask].groupby(keys).relevance.nunique()
n_dup_pairs = len(label_counts)
conflict_keys = set(label_counts[label_counts > 1].index)
print('pairs that appear more than once:', n_dup_pairs)
print('...of which have conflicting labels:', len(conflict_keys))

# Drop conflicting pairs entirely (we cannot know the true label),
# then keep one copy of each remaining duplicate
pairs = pairs[~pairs.set_index(keys).index.isin(conflict_keys)]
pairs = pairs.drop_duplicates(keys).reset_index(drop=True)
print('pairs after cleaning:', len(pairs))

## 5. Leakage check on the original split
If the same job appears in both the original train and test files, a fine-tuned model would be tested on jobs it has already seen. We check this, and then make our own split **grouped by job** so it can't happen.

In [ ]:
orig_tr_jobs = set(pairs.query("source_split == 'train'").job_id)
orig_te_jobs = set(pairs.query("source_split == 'test'").job_id)
orig_tr_res = set(pairs.query("source_split == 'train'").resume_id)
orig_te_res = set(pairs.query("source_split == 'test'").resume_id)
print('jobs in both original train and test:   ', len(orig_tr_jobs & orig_te_jobs))
print('resumes in both original train and test:', len(orig_tr_res & orig_te_res))

## 6. Our split: 70 / 15 / 15, grouped by job (seed 42)

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=SEED)
tr_idx, rest_idx = next(gss.split(pairs, groups=pairs.job_id))
train_df, rest = pairs.iloc[tr_idx], pairs.iloc[rest_idx]

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED)
va_idx, te_idx = next(gss2.split(rest, groups=rest.job_id))
val_df, test_df = rest.iloc[va_idx], rest.iloc[te_idx]

for name, df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    print(f"{name:5}: {len(df):5} pairs | {df.job_id.nunique():3} jobs | "
          f"{df.resume_id.nunique():4} resumes | labels {df.label.value_counts().to_dict()}")

assert not set(train_df.job_id) & set(test_df.job_id)
assert not set(train_df.job_id) & set(val_df.job_id)
print('\nresumes shared between train and test (allowed, reported in paper):',
      len(set(train_df.resume_id) & set(test_df.resume_id)))

## 7. Ranking lists
Each job becomes one ranking query: *"rank these candidates for this job"*. A job is **usable** for ranking metrics if it has at least 5 candidates, at least one relevant one (Potential or Good), and more than one relevance level. Otherwise every ordering scores the same.

In [ ]:
def ranking_table(df):
    g = df.groupby('job_id').agg(
        candidates=('resume_id', 'size'),
        relevant=('relevance', lambda r: int((r > 0).sum())),
        good=('relevance', lambda r: int((r == 2).sum())),
        levels=('relevance', 'nunique'),
    )
    g['usable'] = (g.candidates >= 5) & (g.relevant >= 1) & (g.levels >= 2)
    return g

rank_stats = {}
for name, df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    g = ranking_table(df)
    rank_stats[name] = {
        'jobs': int(len(g)), 'usable_jobs': int(g.usable.sum()),
        'candidates_per_job_median': float(g.candidates.median()),
        'candidates_per_job_min': int(g.candidates.min()),
        'candidates_per_job_max': int(g.candidates.max()),
    }
    print(name, rank_stats[name])

test_df = test_df.assign(eval_usable=test_df.job_id.map(ranking_table(test_df).usable))
val_df = val_df.assign(eval_usable=val_df.job_id.map(ranking_table(val_df).usable))

## 8. Text lengths
These matter for the models. SBERT reads about 256–512 tokens, and LLM prompt size drives the cost.

In [ ]:
lengths = pd.DataFrame({
    'job_words': pairs.drop_duplicates('job_id').job_text.str.split().str.len().describe(),
    'resume_words': pairs.drop_duplicates('resume_id').resume_text.str.split().str.len().describe(),
}).round(0)
lengths

## 9. Save the splits

In [ ]:
train_df.to_parquet(f'{OUT}/train.parquet', index=False)
val_df.to_parquet(f'{OUT}/val.parquet', index=False)
test_df.to_parquet(f'{OUT}/test.parquet', index=False)

summary = {
    'dataset': 'cnamuangtoun/resume-job-description-fit',
    'seed': SEED,
    'pairs_after_cleaning': int(len(pairs)),
    'duplicate_pairs': int(n_dup_pairs),
    'conflicting_pairs_dropped': len(conflict_keys),
    'original_split_job_overlap': len(orig_tr_jobs & orig_te_jobs),
    'splits': rank_stats,
}
with open(f'{OUT}/split_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))

## 10. Blind sample for human labelling (RQ3)
150 pairs from the **test** split, 50 per dataset label, shuffled. **The file does not show the dataset label**, so your judgement is independent.

**How to label:** read the job, then the resume, and pick one of these from the dropdown in the `your_label` column:
- **Good Fit**: meets the main requirements (role, key skills, experience level)
- **Potential Fit**: related background, but missing some key requirements
- **No Fit**: a different field, or clearly unqualified

Don't open `test.parquet` or the key file before you finish. That would spoil the check.

In [ ]:
sample = pd.concat([
    d.sample(n=min(50, len(d)), random_state=SEED) for _, d in test_df.groupby('label')
]).sample(frac=1, random_state=SEED).reset_index(drop=True)
sample.insert(0, 'pair_no', range(1, len(sample) + 1))

MAX_CELL = 32000  # Excel's cell limit is 32,767 characters
sheet = pd.DataFrame({
    'pair_no': sample.pair_no,
    'job_description': sample.job_text.str.slice(0, MAX_CELL),
    'resume': sample.resume_text.str.slice(0, MAX_CELL),
    'your_label': '',
    'note (optional)': '',
})
xlsx_path = f'{OUT}/human_labelling_sample.xlsx'
sheet.to_excel(xlsx_path, index=False)

# Make the sheet readable: wide wrapped columns and a dropdown for the label
from openpyxl import load_workbook
from openpyxl.styles import Alignment
from openpyxl.worksheet.datavalidation import DataValidation
wb = load_workbook(xlsx_path)
ws = wb.active
for col, width in zip('ABCDE', (8, 70, 70, 16, 30)):
    ws.column_dimensions[col].width = width
for row in ws.iter_rows(min_row=2):
    for cell in row:
        cell.alignment = Alignment(wrap_text=True, vertical='top')
dv = DataValidation(type='list', formula1='"Good Fit,Potential Fit,No Fit"', allow_blank=True)
ws.add_data_validation(dv)
dv.add(f'D2:D{len(sheet) + 1}')
ws.freeze_panes = 'B2'
wb.save(xlsx_path)

# The answer key stays separate, for scoring agreement later
sample[['pair_no', 'job_id', 'resume_id', 'label']].to_csv(f'{OUT}/human_labelling_key.csv', index=False)
print('Labelling sheet:', xlsx_path, '|', len(sheet), 'pairs')

## Done
Your Drive folder `resume-job-matching/data` now has:
- `train.parquet`, `val.parquet`, `test.parquet`: the job-grouped splits
- `split_summary.json`: the numbers for the paper's dataset section
- `human_labelling_sample.xlsx`: **the human labelling task** (150 pairs)
- `human_labelling_key.csv`: the dataset labels for the same pairs (don't open it yet)

**Next:** Notebook 2, the baselines (TF-IDF and BM25) and the first results table.